# 01 · Data Cleaning & Unification

Cleans the three raw post-monsoon groundwater datasets (2018–2020) into one tidy,
analysis-ready table and derives a BIS 10500 drinking-water risk label.

This notebook mirrors `src/clean_data.py` step by step so the process is auditable.
The script is the source of truth for the pipeline; run it with `python src/clean_data.py`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
files = {
    2018: RAW / "ground_water_quality_2018_post.csv",
    2019: RAW / "ground_water_quality_2019_post.csv",
    2020: RAW / "ground_water_quality_2020_post.csv",
}

## 1. Inspect the raw headers

The three files describe the same measurements but with inconsistent names,
stray whitespace, and (in 2020) an extra empty column.

In [2]:
for yr, p in files.items():
    cols = pd.read_csv(p, encoding="latin-1", nrows=0).columns.tolist()
    print(yr, cols)

2018 ['sno', 'district', 'mandal', 'village', 'lat_gis', 'long_gis', 'gwl', 'season', 'pH', 'E.C', 'TDS', 'CO3', 'HCO3', 'Cl', 'F', 'NO3 ', 'SO4', 'Na', 'K', 'Ca', 'Mg', 'T.H', 'SAR', 'Classification', 'RSC  meq  / L', 'Classification.1']
2019 ['sno', 'district', 'mandal', 'village', 'lat_gis', 'long_gis', 'gwl', 'season', 'pH', 'EC', 'TDS', 'CO_-2 ', 'HCO_ - ', 'Cl -', 'F -', 'NO3- ', 'SO4-2', 'Na+', 'K+', 'Ca+2', 'Mg+2', 'T.H', 'SAR', 'Classification', 'RSC  meq  / L', 'Classification.1']
2020 ['sno', 'district', 'mandal', 'village', 'lat_gis', 'long_gis', 'gwl', 'season', 'Unnamed: 8', 'pH', 'E.C', 'TDS', 'CO3', 'HCO3', 'Cl', 'F', 'NO3 ', 'SO4', 'Na', 'K', 'Ca', 'Mg', 'T.H', 'SAR', 'Classification', 'RSC  meq  / L', 'Classification.1']


## 2. Canonical column map

Every raw header (after `.strip()`) maps to one clean, consistent name.

In [3]:
COLUMN_MAP = {
    "sno": "sno", "district": "district", "mandal": "mandal", "village": "village",
    "lat_gis": "lat", "long_gis": "lon", "gwl": "gwl", "season": "season", "pH": "ph",
    "E.C": "ec", "EC": "ec", "TDS": "tds",
    "CO3": "co3", "CO_-2": "co3", "HCO3": "hco3", "HCO_ -": "hco3",
    "Cl": "cl", "Cl -": "cl", "F": "f", "F -": "f",
    "NO3": "no3", "NO3-": "no3", "SO4": "so4", "SO4-2": "so4",
    "Na": "na", "Na+": "na", "K": "k", "K+": "k",
    "Ca": "ca", "Ca+2": "ca", "Mg": "mg", "Mg+2": "mg",
    "T.H": "th", "SAR": "sar", "Classification": "irrigation_class",
    "RSC  meq  / L": "rsc", "Classification.1": "rsc_class",
}
NUMERIC_COLS = ["lat","lon","gwl","ph","ec","tds","co3","hco3","cl","f",
                "no3","so4","na","k","ca","mg","th","sar","rsc"]

## 3. Load, drop junk columns, rename, tag the year

In [4]:
def load(path, year):
    df = pd.read_csv(path, encoding="latin-1")
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=COLUMN_MAP)
    df["season"] = "post_monsoon"
    df["year"] = year
    return df

df = pd.concat([load(p, yr) for yr, p in files.items()], ignore_index=True)
print(df.shape)
df.head()

(1106, 27)


,sno,district,mandal,village,lat,lon,gwl,season,ph,ec,...,na,k,ca,mg,th,sar,irrigation_class,rsc,rsc_class,year
0,1,ADILABAD,Adilabad,Adilabad,19.668300,78.524700,5.09,post_monsoon,8.28,745,...,49.0,4.0,48.0,38.896,279.934211,1.273328,C2S1,-1.198684,P.S.,2018
1,2,ADILABAD,Bazarhatnur,Bazarhatnur,19.458888,78.350833,5.10,post_monsoon,8.29,921,...,42.0,5.0,56.0,63.206,399.893092,0.913166,C3S1,-3.397862,P.S.,2018
2,3,ADILABAD,Gudihatnoor,Gudihatnoor,19.525555,78.512222,4.98,post_monsoon,7.69,510,...,45.0,2.0,24.0,38.896,219.934211,1.319284,C2S1,-0.398684,P.S.,2018
3,4,ADILABAD,Jainath,Jainath,19.730555,78.640000,5.75,post_monsoon,8.09,422,...,27.0,1.0,32.0,19.448,159.967105,0.928155,C2S1,0.000658,P.S.,2018
4,5,ADILABAD,Narnoor,Narnoor,19.495665,78.852654,2.15,post_monsoon,8.21,2321,...,298.0,5.0,56.0,92.378,519.843750,5.682664,C4S2,-4.396875,P.S.,2018


## 4. Coerce numeric, clean text, impute groundwater level

In [5]:
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ["district", "mandal", "village"]:
    df[c] = df[c].astype(str).str.strip().str.title()
for c in ["irrigation_class", "rsc_class"]:
    df[c] = df[c].astype(str).str.strip().str.upper()

# groundwater level: district median, then global median
df["gwl"] = df["gwl"].fillna(df.groupby("district")["gwl"].transform("median"))
df["gwl"] = df["gwl"].fillna(df["gwl"].median())

df[NUMERIC_COLS].isna().sum()

lat       0
lon       0
gwl       0
ph        1
ec        0
tds       0
co3     160
hco3      0
cl        0
f         0
no3       0
so4       0
na        0
k         0
ca        0
mg        0
th        0
sar       0
rsc       0
dtype: int64

## 5. Derive BIS 10500 drinking-water risk

Count how many acceptable limits each sample exceeds, then bin into
Safe (0) / Moderate (1–2) / High (3+).

In [6]:
BIS = {"ph_low":6.5,"ph_high":8.5,"tds":500,"th":200,"cl":250,
       "f":1.0,"no3":45,"so4":200,"ca":75,"mg":30}

def exceedances(r):
    n = 0
    if pd.notna(r["ph"]) and not (BIS["ph_low"] <= r["ph"] <= BIS["ph_high"]):
        n += 1
    for k in ["tds","th","cl","f","no3","so4","ca","mg"]:
        if pd.notna(r[k]) and r[k] > BIS[k]:
            n += 1
    return n

df["bis_exceedances"] = df.apply(exceedances, axis=1)
df["drinking_risk"] = df["bis_exceedances"].apply(
    lambda n: "Safe" if n == 0 else ("Moderate" if n <= 2 else "High"))

df["drinking_risk"].value_counts()

drinking_risk
High        841
Moderate    176
Safe         89
Name: count, dtype: int64

## 6. Save the cleaned dataset

In [7]:
out = Path("../data/processed/groundwater_clean_2018_2020.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print("Saved", out, "->", df.shape)

Saved ../data/processed/groundwater_clean_2018_2020.csv -> (1106, 29)
